# Multi-Weather Evaluation

This notebook applies the full pipeline to multiple adverse-weather conditions: snow, rain and fog. It uses two weather reference images per condition to evaluate how reference-image choice affects generated outputs.


In [ ]:
!pip -q install -r /content/image-data-generation/requirements.txt

In [ ]:
import os
import time
import gc
from pathlib import Path

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

In [ ]:
repo_dir = Path("/content/image-data-generation")

dataset_dir = repo_dir / "dataset" / "source_road_scenes"
reference_dir = repo_dir / "dataset" / "weather_reference_conditions"
results_dir = repo_dir / "results" / "multi_weather_evaluation"

results_dir.mkdir(parents=True, exist_ok=True)

if not dataset_dir.exists():
    raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

if not reference_dir.exists():
    raise FileNotFoundError(f"Reference folder not found: {reference_dir}")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Dataset folder:", dataset_dir)
print("Reference folder:", reference_dir)
print("Results folder:", results_dir)

In [ ]:
valid_exts = (".jpg", ".jpeg", ".png", ".jfif", ".webp")

image_paths = sorted([
    p for p in dataset_dir.iterdir()
    if p.is_file() and p.name.lower().endswith(valid_exts)
])

if len(image_paths) == 0:
    raise FileNotFoundError("No source road-scene images found.")

print(f"Found {len(image_paths)} source road-scene images.")

weather_configs = {
    "snow": {
        "prompt": "Convert image to snowy winter with heavy snowfall",
        "references": [
            reference_dir / "snow_reference_1.jpg",
            reference_dir / "snow_reference_2.jpg"
        ]
    },
    "rain": {
        "prompt": "Convert image to a rainy road scene with wet roads and rainfall",
        "references": [
            reference_dir / "rain_reference_1.jpg",
            reference_dir / "rain_reference_2.jpg"
        ]
    },
    "fog": {
        "prompt": "Convert image to a foggy road scene with reduced visibility",
        "references": [
            reference_dir / "fog_reference_1.jpg",
            reference_dir / "fog_reference_2.jpg"
        ]
    }
}

for weather_name, config in weather_configs.items():
    for ref_path in config["references"]:
        if not ref_path.exists():
            raise FileNotFoundError(f"Missing reference image: {ref_path}")

In [ ]:
print("Loading BLIP-2...")

blip_dtype = torch.float16 if device == "cuda" else torch.float32

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=blip_dtype,
    device_map="auto"
)

captions = {}

for image_path in image_paths:
    image_id = image_path.stem
    image = Image.open(image_path).convert("RGB")

    inputs = processor(image, return_tensors="pt").to(blip_model.device)

    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_length=40)

    caption = processor.decode(ids[0], skip_special_tokens=True)

    if not caption or len(caption.strip()) == 0:
        raise ValueError(f"BLIP-2 did not generate a valid caption for {image_id}")

    captions[image_id] = caption
    print(image_id, ":", caption)

del blip_model
del processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print("Loading MiDaS...")

midas_device = torch.device(device)

midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

In [ ]:
print("Loading ControlNet...")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

print("Loading Stable Diffusion + ControlNet...")

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)

print("Loading IP-Adapter...")

pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

pipe.set_ip_adapter_scale(0.3)

In [ ]:
negative_prompt = (
    "blurry, distorted cars, distorted road, bad geometry, "
    "low quality, warped perspective, artifacts"
)

MAX_IMAGES = None
SKIP_EXISTING = True # useful if Colab disconnects

run_image_paths = image_paths[:MAX_IMAGES] if MAX_IMAGES else image_paths

required_files = [
    "original.png",
    "reference_condition.png",
    "depth_gray.png",
    "generated_output.png",
    "comparison.png",
    "metadata.txt"
]

for image_path in run_image_paths:
    image_id = image_path.stem
    print(f"\nProcessing source image: {image_id}")

    image = Image.open(image_path).convert("RGB")
    init_image = image.resize((512, 512))

    # Generate MiDaS depth map once per source image
    img_np = np.array(image)
    input_batch = transform(img_np).to(midas_device)

    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze(0).squeeze(0)

    depth = prediction.detach().cpu().numpy()
    depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
    depth_vis = depth_vis.astype(np.uint8)

    depth_gray = Image.fromarray(depth_vis).convert("RGB").resize((512, 512))

    for weather_name, config in weather_configs.items():
        user_prompt = config["prompt"]

        final_prompt = (
            f"Scene description: {captions[image_id]}. "
            f"User prompt: {user_prompt}. "
            "Preserve original road layout, vehicles, object positions, and perspective."
        )

        for ref_index, ref_path in enumerate(config["references"], start=1):
            output_dir = results_dir / image_id / f"{weather_name}_ref{ref_index}"
            output_dir.mkdir(parents=True, exist_ok=True)

            if SKIP_EXISTING and all((output_dir / file).exists() for file in required_files):
                print(f"Skipping complete run: {image_id} | {weather_name} | ref{ref_index}")
                continue

            reference_image = Image.open(ref_path).convert("RGB").resize((512, 512))

            init_image.save(output_dir / "original.png")
            reference_image.save(output_dir / "reference_condition.png")
            depth_gray.save(output_dir / "depth_gray.png")

            generator = torch.Generator(device=device).manual_seed(42)

            print(f"Generating {image_id} | {weather_name} | ref{ref_index}")

            start_time = time.time()

            generated = pipe(
                prompt=final_prompt,
                negative_prompt=negative_prompt,
                image=init_image,
                control_image=depth_gray,
                ip_adapter_image=reference_image,
                strength=0.8,
                guidance_scale=7.5,
                controlnet_conditioning_scale=1.0,
                num_inference_steps=30,
                generator=generator
            ).images[0]

            runtime = time.time() - start_time

            generated.save(output_dir / "generated_output.png")

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))

            axes[0].imshow(init_image)
            axes[0].set_title("Original")
            axes[0].axis("off")

            axes[1].imshow(reference_image)
            axes[1].set_title("Weather Reference")
            axes[1].axis("off")

            axes[2].imshow(depth_gray)
            axes[2].set_title("MiDaS Depth")
            axes[2].axis("off")

            axes[3].imshow(generated)
            axes[3].set_title(f"{weather_name.capitalize()} Output")
            axes[3].axis("off")

            plt.tight_layout()
            plt.savefig(output_dir / "comparison.png", dpi=300)
            plt.show()
            plt.close()

            with open(output_dir / "metadata.txt", "w") as f:
                f.write(f"Image ID: {image_id}\n")
                f.write(f"Input path: {image_path}\n")
                f.write(f"Weather condition: {weather_name}\n")
                f.write(f"Reference image: {ref_path}\n")
                f.write(f"Reference index: {ref_index}\n")
                f.write(f"BLIP-2 caption: {captions[image_id]}\n")
                f.write(f"User prompt: {user_prompt}\n")
                f.write(f"Final prompt: {final_prompt}\n")
                f.write(f"Negative prompt: {negative_prompt}\n")
                f.write("Seed: 42\n")
                f.write("Image size: 512x512\n")
                f.write("Strength: 0.8\n")
                f.write("Guidance scale: 7.5\n")
                f.write("Inference steps: 30\n")
                f.write("ControlNet conditioning scale: 1.0\n")
                f.write("IP-Adapter scale: 0.3\n")
                f.write(f"Runtime: {runtime:.2f} seconds\n")

            missing_files = [
                file for file in required_files
                if not (output_dir / file).exists()
            ]

            if missing_files:
                print("Missing files:", missing_files)
            else:
                print("All expected files saved for", image_id, weather_name, f"ref{ref_index}")

            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
expected_runs = len(image_paths) * 3 * 2
complete_runs = 0
missing_runs = []

for image_path in image_paths:
    image_id = image_path.stem

    for weather_name in weather_configs.keys():
        for ref_index in [1, 2]:
            run_dir = results_dir / image_id / f"{weather_name}_ref{ref_index}"

            missing = [
                file for file in required_files
                if not (run_dir / file).exists()
            ]

            if missing:
                missing_runs.append((str(run_dir), missing))
            else:
                complete_runs += 1

print("Expected runs:", expected_runs)
print("Complete runs:", complete_runs)
print("Missing/incomplete runs:", len(missing_runs))

if missing_runs:
    for run_dir, missing in missing_runs[:20]:
        print(run_dir, "missing:", missing)
else:
    print("All multi-weather outputs are complete.")

In [ ]:
del pipe
del controlnet
del midas_model
del midas_transforms
del transform

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")